In [5]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------------------
# 1. Sharpe Ratio Function
# -----------------------------------------
def calc_spread_return_sharpe(df: pd.DataFrame, portfolio_size: int = 200, toprank_weight_ratio: float = 2) -> float:
    def _calc_spread_return_per_day(g, portfolio_size, toprank_weight_ratio):
        assert g['Rank'].min() == 0
        assert g['Rank'].max() == len(g['Rank']) - 1
        n_stocks = len(g)
        ps = min(portfolio_size, n_stocks)
        top = g.sort_values(by='Rank')['Target'][:ps]
        bot = g.sort_values(by='Rank', ascending=False)['Target'][:ps]
        w = np.linspace(start=toprank_weight_ratio, stop=1, num=len(top))
        ret_long  = (top  * w).sum() / w.mean()
        ret_short = (bot  * w).sum() / w.mean()
        return ret_long - ret_short

    daily_spreads = df.groupby('Date').apply(_calc_spread_return_per_day,
                                             portfolio_size, toprank_weight_ratio)
    return daily_spreads.mean() / (daily_spreads.std() + 1e-8)

# -----------------------------------------
# 2. Load and preprocess data
# -----------------------------------------
data = pd.read_csv("augment_Date.csv")
data = (
    data
    .dropna()
    .sort_values(["Date", "SecuritiesCode"])
    .reset_index(drop=True)
)

# -----------------------------------------
# 3. Split into train and test
# -----------------------------------------
split_idx = int(len(data) * 0.8)
train_df = data.iloc[:split_idx].reset_index(drop=True)
test_df  = data.iloc[split_idx:].reset_index(drop=True)

# -----------------------------------------
# 4. Filter: keep only samples with non-zero sentiments
# -----------------------------------------
def filter_nonzero_sentiment(df):
    return df[~((df['sentiment_score_all'] == 0) & (df['sentiment_score_sub'] == 0))].reset_index(drop=True)

#train_df = filter_nonzero_sentiment(train_df)
#test_df  = filter_nonzero_sentiment(test_df)

# -----------------------------------------
# 5. Define sentiment boosting (scaling + duplication)
# -----------------------------------------
X_train_sent = train_df.copy()
X_test_sent  = test_df.copy()
sentiment_cols = ['sentiment_score_all', 'sentiment_score_sub']
scale_factor = 1
dup_times = 1

# Scale + duplicate
for feat in sentiment_cols:
    X_train_sent[feat] *= scale_factor
    X_test_sent[feat]  *= scale_factor
    for i in range(dup_times):
        X_train_sent[f"{feat}_dup{i}"] = X_train_sent[feat]
        X_test_sent[f"{feat}_dup{i}"]  = X_test_sent[feat]

# Define features
features_sent_boosted = [c for c in X_train_sent.columns if c not in ['Target', 'Date']]
y_train = train_df['Target']
y_test  = test_df['Target']

# -----------------------------------------
# 6. Define no-sentiment version
# -----------------------------------------
X_train_nosent = train_df.drop(columns=sentiment_cols)
X_test_nosent  = test_df.drop(columns=sentiment_cols)
features_nosent = [c for c in X_train_nosent.columns if c not in ['Target', 'Date']]

# -----------------------------------------
# 7. Hyperparameters
# -----------------------------------------
best_params_v2 = {
    "boosting_type": "gbdt",
    "num_leaves": 77,
    "max_depth": 19,
    "min_child_samples": 41,
    "learning_rate": 0.09953003012549799,
    "n_estimators": 2000,
    "colsample_bytree": 0.42197751965634284,
    "reg_alpha": 2.0880923278589756,
    "reg_lambda": 3.073547390938834,
    "subsample": 0.8832492105102278,
    "bagging_freq": 1,
    "random_state": 42,
    "metric": "mae"
}

# -----------------------------------------
# 8. Model A: Boosted Sentiment
# -----------------------------------------
model_sent = LGBMRegressor()
model_sent.fit(X_train_sent[features_sent_boosted], y_train)

y_pred_sent = model_sent.predict(X_test_sent[features_sent_boosted])
rmse_sent = np.sqrt(mean_squared_error(y_test, y_pred_sent))
mae_sent  = mean_absolute_error(y_test, y_pred_sent)

test_eval_sent = test_df.copy()
test_eval_sent['pred'] = y_pred_sent
test_eval_sent['Rank'] = test_eval_sent.groupby('Date')['pred'].rank(method='first', ascending=False) - 1
sharpe_sent = calc_spread_return_sharpe(test_eval_sent)

# -----------------------------------------
# 9. Model B: No Sentiment
# -----------------------------------------
model_nosent = LGBMRegressor()
model_nosent.fit(X_train_nosent[features_nosent], y_train)

y_pred_nosent = model_nosent.predict(X_test_nosent[features_nosent])
rmse_nosent = np.sqrt(mean_squared_error(y_test, y_pred_nosent))
mae_nosent  = mean_absolute_error(y_test, y_pred_nosent)

test_eval_nosent = test_df.copy()
test_eval_nosent['pred'] = y_pred_nosent
test_eval_nosent['Rank'] = test_eval_nosent.groupby('Date')['pred'].rank(method='first', ascending=False) - 1
sharpe_nosent = calc_spread_return_sharpe(test_eval_nosent)
print("=== Model A: Boosted Sentiment ===")
print(f"RMSE: {rmse_sent:.4f}, MAE: {mae_sent:.4f}, Sharpe: {sharpe_sent:.4f}")
print("=== Model B: No Sentiment ===")
print(f"RMSE: {rmse_nosent:.4f}, MAE: {mae_nosent:.4f}, Sharpe: {sharpe_nosent:.4f}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000380 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5507
[LightGBM] [Info] Number of data points in the train set: 625, number of used features: 31
[LightGBM] [Info] Start training from score 0.000540
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

C:\Users\zixuanmu\AppData\Local\Temp\1\ipykernel_10620\3496189938.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_spreads = df.groupby('Date').apply(_calc_spread_return_per_day,
C:\Users\zixuanmu\AppData\Local\Temp\1\ipykernel_10620\3496189938.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_spreads = df.groupby('Date').apply(_calc_spread_return_per_day,
